# Assignment 2 -  Extended Kalman filter and Unscented Kalman filter


- **Topic:** Desiging an EKF and UKF to estimate the position using magnetic field information
- **Assessment:** The assignment will go through a pass/fail check.


- **Deadline:**  06-03-2026, 17:00
- **Submitting: SUBMIT ONLY `assignment2_groupNumber.ipynb` TO BRIGHTSPACE**, where groupNumber is your groupnumber



## Instructions
**Installation:** The implementation is tested with python 3.9.12 and packages including
-   numpy version: 1.26.4
-   scipy version: 1.13.0
-   matplotlib version: 3.5.1


Other (not too old or new) versions will probabily also work. For windows user check the announcement (**"Fix of Issues Assignment 1"**) on brightspace which could also be helpful for this assignment.

You may not use other packages for algorithm-related calculations.
You only need to complete (and submit) this file.
Please do not change the additional files `GP.py`, `helper.py` and `linAlg.py` and the dictionary `modelParameters` as this might result in breaking the assignment.



## AI Related Policy
We strongly discourage you to use AI tools for implementation assistance. It is your understanding of the problem that is tested in the final exam.


## Information
Please fill in your group number, names and student numbers in the cell below.

<!-- first EKF and then UKF, very similar to normal KF-->

In [ ]:
''' YOUR ANSWER HERE '''
groupNumber = 22

STUDENT_1_NAME = "Santiago Terreno Duarte"
STUDENT_1_STUDENT_NUMBER = "6144683"

STUDENT_2_NAME = "Pragun Srivastava"
STUDENT_2_STUDENT_NUMBER = "5542707"

# raise NotImplementedError()

## Objectives

#### Goal

The goal of this assignment is to estimate the position $ \mathbf{x}_k $ of an agent, the same as assignment 1, but using different algorithmns. The agent moves in an indoor environment and gets information about its approximate change in position $ \Delta \mathbf{x}_k $ at every time-step. To model this, you may assume a linear dynamic model of the form:
$$\mathbf{x}_{k + 1} = \mathbf{x}_k + \Delta \mathbf{x}_{k} + \mathbf{w}_{k}$$

Where:

- $\mathbf{x}_k$ is a 3-dimensional position vector of the agent at time-step $k$. 

- $\Delta \mathbf{x}_k$ is the change in positon.

- $\mathbf{w}_k \sim \mathcal{N}(0_{3 \times 1}, \mathbf{Q})$ is Gaussian white noise.

- $ \mathbf{Q} \in \mathbb{R}^{3 \times 3} $ is the covariance of the process noise.

Since the information of the change in position $ \Delta \mathbf{x}_k$ contains errors, the position estimates from the dynamic model will drift. This means that over time, the agent will lose track of where it actually is. To remedy this, the agent is carrying a magnetometer, and has a map of the magnetic field. The magnetometer samples a measurement of the magnetic field $\mathbf{y}_k$ at every position $\mathbf{x}_k$. The map of the magnetic field is modeled by a non-linear function: a reduced rank Gaussian process (you do not need to understand, model or change the GP model, only to call the function evaluation). This results in the non-linear measurement model:
$$ \mathbf{y}_k = f(\mathbf{x}_k) + \mathbf{v}_k $$

Where:

- $\mathbf{y}_k$ is a 3-dimensional measurement vector of the magnetic field at time-step $k$.

- $f(\cdot)$ is the reduced rank Gaussian process function.

- $\mathbf{v}_k \sim \mathcal{N}(0_{3 \times 1}, \mathbf{R})$ is Gaussian white noise.

- $ \mathbf{R} \in \mathbb{R}^{3 \times 3} $ is the covariance of the measurement noise.


#### First load in the data
To start, first load in the data below
All variables you will use are numpy arrays, where
- `magnetometerMeasurements` $\in \mathbb{R}^{3 \times \text{K}}$ is a numpy array containing the magnetometer measurements
- `magnetometerInitialPosition` $\in \mathbb{R}^{3 \times 1}$ is a numpy array containing the initial position of the magnetometer
- `deltaMagnetometerPositions` $\in \mathbb{R}^{3 \times \text{K}}$ is a numpy array containing the change in position of the magnetometer

#### Warning
- Additionally you are provided with a dictionary modelParameters which is used in creating magnetic field maps. DO NOT CHANGE dictionary as this might break the assignment. To create the EKF and UKF you DO NOT need this dictionary.

In [ ]:
''' DO NOT CHANGE THE CODE BELOW '''
import numpy as np
import matplotlib.pyplot as plt

import GP as GP

import linAlg as linAlg
import helper as helper
np.random.seed(groupNumber)

(magnetometerMeasurements, 
 magnetometerInitialPosition, 
 deltaMagnetometerPositions,
 modelParameters) = helper.initializeKalmanFilterAssignment(groupNumber % 29)


Same as the first assignment of particle filter, below, a function `helper.makeDeadReckoningPlots()` is given to plot the map of the magnetic field and the ground truth position of the path taken by the agent. The agent does not know this path, it only has access to its initial position, the change in position at every time-step and the magnetometer measurements. You can reuse the code for the plots.

Your first task it to use the agent's dynamic model to try and reconstruct the ground truth path. This is also known as dead reckoning and should be done using data stored in the variables `magnetometerInitialPosition` and `deltaMagnetometerPositions`. Save the agent's positions according to the dynamic model in the variable `deadReckoning` $\in \mathbb{R}^{3 \times \text{K}}$. Additionally, compare the magnetometer measurements with the map of the magnetic field. To achieve this, store the norm of the magnetic field at every time-step in the variable `magneticFieldNorm` $\in \mathbb{R}^{\text{K}}$.

This function should output two subfigures. Every subfigure includes the magnetic field map in the background. On the left the ground truth positions are plotted together with the dead reckoning positions. On the right, the magnetic field measurements are plotted with  the ground truth positions over the magnetic field map.

In [ ]:
T = deltaMagnetometerPositions.shape[1]
cum_deltas = np.zeros([3,T])
cum_delta = np.zeros(3)
for i in range(T):
    cum_delta += deltaMagnetometerPositions[:,i]
    cum_deltas[:,i] = cum_delta

deadReckoning = magnetometerInitialPosition + cum_deltas
magneticFieldNorm = np.linalg.norm(magnetometerMeasurements, axis = 0)


helper.makeDeadReckoningPlots(deadReckoning, magneticFieldNorm, modelParameters)

## EKF and UKF
To  use magnetometer measurements to reduce the positional drift observed in the figure above, besides PF in the previous lectures, you can also use EKF and UKF.

### EKF Implementation

You start from implementing EKF.  Your task is to fill in the following functions.

#### Task 1
Fill in the `dynamicUpdateEKF()` function according to the dynamic model given above. 

**Note:**
You do not have to change the inputs and outputs to the given functions.


In [ ]:

def dynamicUpdateEKF(position, 
                   deltaPos, 
                   posCov, 
                   motionCovarianceEKF, 
                   modelParameters):
    ''' Dynamic update '''
    position = position + deltaPos
    posCov = posCov + motionCovarianceEKF

    return position, posCov



#### Task 2

Complete the function `measurementUpdateEKF()` according to the measurement model given above. 

To implement the measurement model, you will need to make predictions of the magnetic field. You DO NOT need to create this function yourself. To make predictions, use the function `GP.makeMagneticFieldPrediction(positions, modelParameters)` function make sure the input `positions` $\in \mathbb{R}^{3 \times 1} $. The output should be a matrix $\in \mathbb{R}^{3 \times 1}$ magnetic field predictions for every input location.  Similarly, to find the Jacobian, you muse use the GP.makeMagneticFieldJacobian(position, modelParameters) function make sure the input position is a vector of 3x1. The output should be a matrix of 3x3.

In [ ]:
def measurementUpdateEKF(position, 
                         magnetometerMeasurement, 
                         posCov, 
                         measurementCovarianceEKF,
                         modelParameters):
    
    ''' Measurement update '''

    magneticFieldPrediction = GP.makeMagneticFieldPrediction(position, modelParameters) 
    jacobianMagneticFieldPrediction = GP.makeMagneticFieldJacobian(position, modelParameters)

    '''
    fill in your code
    '''
    H = jacobianMagneticFieldPrediction
    R = measurementCovarianceEKF
    innovation = magnetometerMeasurement - magneticFieldPrediction

    S = H @ posCov @ H.T + R
    K = posCov @ H.T @ np.linalg.inv(S)
    
    position = position + K @ innovation
    posCov = posCov - K @ H @ posCov

    return position, posCov

`extendedKalmanFilter()` is given as follows. You don't need to change it.

In [ ]:
''' DO NOT CHANGE THE CODE BELOW '''

def extendedKalmanFilter(position, 
                         deltaPosition, 
                         magnetometerMeasurement, 
                         posCov, 
                         motionCovarianceEKF, 
                         measurementCovarianceEKF, 
                         modelParameters):
    ''' Estimates positions'''

    ''' Dynamic update '''    
    if modelParameters['NtimeStepEKF'] > 0:
        (position, 
         posCov) = dynamicUpdateEKF(position, 
                                  deltaPosition, 
                                  posCov, 
                                  motionCovarianceEKF,
                                  modelParameters)

    ''' Measurement update '''
    (position, 
     posCov) = measurementUpdateEKF(position,
                                    magnetometerMeasurement, 
                                    posCov, 
                                    measurementCovarianceEKF,
                                    modelParameters)

    return position, posCov


#### Task 3

Be sure to set the **initial position** and the **prior** on it correctly  and a **suitable measurement covariance** as you did in assignment 1.

You could start with the following parameters
- `motionCovarianceUKF` = np.eye(3)*0.001**2 
- `measurementCovarianceUKF` = np.eye(3)*1.5**2

Make sure `posCovarianceUKF` is a 3d tensor with the shape of `[3,3,NtimeSteps]`, to save the position covariances for every time step.
To get better results, you might have to tune these. 

In [ ]:
''' Initialize the EKF '''
modelParameters['NtimeStepEKF'] = 0

motionCovarianceEKF = np.eye(3)*0.001**2
measurementCovarianceEKF = np.eye(3)*4**2

N = modelParameters['NtimeSteps']

posEstEKF = np.zeros([3, N])
posEstEKF[:,0:1] = magnetometerInitialPosition

posCovarianceEKF = np.zeros([3, 3, N])
posCovarianceEKF[:, :, 0] = np.eye(3)*0.015**2

''' Run the EKF'''

for timendx in range(modelParameters['NtimeSteps']-1):
    deltaPosEKF = deltaMagnetometerPositions[:, timendx:timendx+1]
    ymeasEKF = magnetometerMeasurements[:, timendx:timendx+1]
    (posEstEKF[:, timendx+1:timendx+2], 
     posCovarianceEKF[:, :, timendx+1]) = extendedKalmanFilter(posEstEKF[:, timendx:timendx+1], 
                                              deltaPosEKF, 
                                              ymeasEKF, 
                                              posCovarianceEKF[:, :, timendx],
                                              motionCovarianceEKF,
                                              measurementCovarianceEKF,
                                              modelParameters)
    modelParameters['NtimeStepEKF'] += 1

#### discussion

Elaborate on the effects of tuning the different variables:

- `motionCovarianceEKF`
- `measurementCovarianceEKF`
- The initial posiiton covariance `posCovarianceEKF[:, :, 0]`


## Answer

motionCovarianceEKF: The motion covariance Q indicates how much we trust our dynamic model. If we believe that our dynamic model is accurate and are reasonably confident about the deltas, we can set it to be relatively low. Conversely, if we are not confident about the dynamics, we should set it high in order to trust the measurments more, because what will determine how the model behaves is the relation bewtween Q and the meaasurement covariance R (more on this at the end of this paragraph). In the extreme case of a very small value for Q relatively to R, the EKF trajectory will drift away just like the dead reckoning, because it will almost entirely trust the dynamics, and if Q and the initial position covariance are both set to zero, then the EKF trajectory will be the same as dead reckoning, since this will make the Kalman gain zero and thus completely ignore the measurements. In the other extreme, a very high value of Q will make the model trust the measurements much more than the dynamics, and consequently the trajectory will have big jumps all over the map trying to match the measured magnetic field. For this particular dataset, it was found that the suggested value for Q worked well (but adjusting the value of R improved the estimates, as we will explain next). Finally, it must be said that although the trajectory will be deterimned by the relation between Q and R, their absolute values still matter to determine the covariance of the estimates. This is important because it gives us information about how certain we are about the estimated positions. Specifically, even though it was found that decreasing Q and R proportionally from the shown final values gave similar estimates from the trajectory, this would make the position covariance showun in the plot decrese more than the error between estimate and true position, which doesn't represent the uncertainty well.

measurementCovarianceEKF: The measurement covariance R indicates how much we trust the measurements and the measurement model. The suggested value turned out to be too low, because the EKF trajectory deviated too much from the square trajectory attempting to match the measured magnetic field, which was particularly acute on the right and bottom sides, since in those parts the measured magnetic field is more distorted and corresponds to the true magetic field closer to the center (higher value). This was already observed in the first assignment, and can be clearly seen in its first plot. Therefore, the value of R was progresively increased to increase the relative trust on the dynamics, and the final value shown is believed to be satisfactory. We could have instead decreased the value of Q, but again that would yield unrealistically low covariances that do not reflect the error in the estimation. Similarly to the cse of Q, if the value of R is too low, there is an excessive trust on the measurements and the EKF trajectory has sudden jumps all over the map trying to match them, and a very high value ignores the measurements and makes the EKF trajectory drift away like the dead reckoning.


initial position covariance: The initial position covariance P0 indicates how much we trust the estimated initial position, set in this case to be the measured magnetometerInitialPosition. If we have no clue about the initial position, a high value for P0 is appropiate, since this would make the model trust the measurments a lot at the beginning and quickly drift away from the initial position and approximate the true trajectory. However, we have a good estimate of the initial position because we have a measurement for it, so a low value for P0 works well since it makes the model deviate from the intial position more slowly because the value of the Kalman gain will be lower at the beginning. In this particular dataset we could even set the value of P0 to 0 since, from the plot, it appears that the measured initial position is really close to the real initial position, if it is not actually the same. But realistically we should still set a non-zero value, even if it is very small, to ensure the measurement updates plays the role from the beginning.

#### Visualize the output
<!-- TODO: keep or not. maybe comparison between EKF and UKF in the end is enough -->

In [ ]:
''' DO NOT CHANGE THE CODE BELOW '''
helper.makeExtendedKalmanFilterPlots(deadReckoning, posEstEKF, posCovarianceEKF, modelParameters)

**Discussion**

Explain why the covariance changes over time the way it does. You can create more plots if necessary for your analysis.

## Answer

The initial covariance is the one that was manually chosen, and then the covariance rapidly decreases and reaches a steady state for the 3 axes. After that the same pattern is repeated as the time step increases (for clarity we show the covariance without the initial values to make the pattern clearly visible). Along all the axis there is a certain periodicity, but the value along the y-axis is the one that changes more dramatically (having higher amplitude), while also having higher period. 

The periodicity can be explained by the fact that in every time update the covariance automatically increases due to the addition of the value of Q (and more generally due to the term F @ P @ F.T, not present here), and in the measurement update it decreases. This makes intuitive sense because every time we do a time update, we are adjusting the position according to the dynamics, whose model has an error given by Q. However, in the measurement update we are doing filtering, not prediction, meaning that we are estimating the current position and confirming via the measurements the value given by the time update, decreasing the covariance due to the presence of new information. If this new information is not useful, then the covariance would decrease very little during the measurement update (it never decreases at this step). This would mean that overall the covariance will increase as time passes, which is exactly what happens in the dead reckoning (as time passes, we are less and less sure about the position). To sum up, after an initial decrease due to the high chosen initial value, the covariance goes up and down persistently since the time update increases it and the measurement update decreases it. 

When the covariance along a certain axis increases step after step it means that the increase of the time update is dominating over the decrease of the measurement update, and vice versa. 

As to why the amplitude and period is higher for the y-axis, it may be because the magnetic field changes less across that axis, and therefore the tendency is slower to change from increasing to decreasing covariance. As a result, the covariance in the y-axis grows more during prediction steps and is reduced less during measurement updates, producing the larger amplitude and longer period observed in the plot. In regions where the magnetic field varies strongly with position, the measurements provide more information about the state, leading to stronger covariance reductions. In contrast, if the magnetic field varies slowly along a particular direction, the measurements are less informative in that direction, so the covariance tends to grow more before being reduced.

In [ ]:
t = np.arange(20, N)

plt.figure()

plt.plot(t, posCovarianceEKF[0,0,20:], color = 'blue', label='Covariance X')
plt.plot(t, posCovarianceEKF[1,1,20:], color = 'green', label='Covariance Y')
plt.plot(t, posCovarianceEKF[2,2,20:], color = 'red', label='Covariance Z')

plt.xlabel('Time step')
plt.ylabel('Covariance')
plt.legend()
plt.grid()

### UKF implement 

#### Task 1
Fill in the `dynamicUpdateUKF()` function according to the dynamic model given above. 

#### Note:
You do not have to change the inputs and outputs to the given functions.


In [ ]:
def dynamicUpdateUKF(position, 
                   deltaPos, 
                   posCov, 
                   motionCovarianceUKF, 
                   modelParameters):
    ''' Dynamic update '''
    position = position + deltaPos
    posCov = posCov + motionCovarianceUKF

    return position, posCov

#### Task 2

- Complete creating sigma points `createSigmaPoints()` and calculating sigma weights `calculateSigmaWeights()`.
- Fill in the `measurementUpdateUKF()` according to the measurement model given above. 
- You can play around with UT1 and UT2, can you see any obvious differences?


In [ ]:
def createSigmaPoints(stateVector, 
                      stateCovariance, 
                      lamdaWeights,
                      modelParameters):
    ''' Create sigma points '''
    n = stateVector.size
    sigmaPoints = np.zeros([n, 2*n + 1])

    sigmaPoints[:,0:1] = stateVector
    
    U, S, Vh = np.linalg.svd(stateCovariance, full_matrices=True, compute_uv=True, hermitian=True)
    sqrtS = np.diag(np.sqrt(S))
    sqrtP = U @ sqrtS @ Vh

    # sqrtP = np.linalg.cholesky(P)

    for i in range(n):
        
        sigmaPoints[:, i+1:i+2] = stateVector + np.sqrt(n + lamdaWeights) * sqrtP[:, i:i+1]

        sigmaPoints[:, i+n+1:i+n+2] = stateVector - np.sqrt(n + lamdaWeights) * sqrtP[:, i:i+1]

    return sigmaPoints


def calculateSigmaWeights(UTnumber = 1):
    n = 3
    ''' Calculate the weights '''
    if UTnumber == 1:
        '''UT1'''

        lamdaWeights = 3 - n

        a = 0

        weightsMean = (0.5/(n + lamdaWeights)) * np.ones(2*n + 1)
        weightsMean[0] = lamdaWeights/(n + lamdaWeights)
        
        weightsCov = (0.5/(n + lamdaWeights)) * np.ones(2*n + 1)
        weightsCov[0] = lamdaWeights/(n + lamdaWeights) + a
        

    else:
        '''UT2'''
        
        lamdaWeights = n*(1e-6 - 1)

        a = 3 - 1e-6

        weightsMean = (0.5/(n + lamdaWeights)) * np.ones(2*n + 1)
        weightsMean[0] = lamdaWeights/(n + lamdaWeights)
        
        weightsCov = (0.5/(n + lamdaWeights)) * np.ones(2*n + 1)
        weightsCov[0] = lamdaWeights/(n + lamdaWeights) + a
        
    return lamdaWeights, weightsMean, weightsCov


def measurementUpdateUKF(position,
                         magnetometerMeasurement,
                         posCovariance,
                         measurementCovarianceUKF,
                         lamdaWeights, 
                         weightsMean, 
                         weightsCovariance,
                         modelParameters):
    
    ''' Measurement update '''
    sigmaPoints = createSigmaPoints(position, 
                      posCovariance, 
                      lamdaWeights,
                      modelParameters)

    
    n = 3
    C = np.zeros([n, n])
    S = np.zeros([n, n])
    y = np.zeros([n,1])
    
    for i in range(2*n + 1):

        xi = sigmaPoints[:,i:i+1]

        propagatedSigmaPoint = GP.makeMagneticFieldPrediction(xi, modelParameters) 
        y += weightsMean[i] * propagatedSigmaPoint

    for i in range(2*n + 1):

        xi = sigmaPoints[:,i:i+1]

        propagatedSigmaPoint = GP.makeMagneticFieldPrediction(xi, modelParameters)
        C += weightsCovariance[i] * ((xi - position) @ (propagatedSigmaPoint - y).T)
        S += weightsCovariance[i] * ((propagatedSigmaPoint - y) @ (propagatedSigmaPoint - y).T)
                                     
    S += measurementCovarianceUKF

    K = C @ np.linalg.inv(S)

    position = position + K @ (magnetometerMeasurement - y)

    posCovariance = posCovariance - K @ S @ K.T
        
    return position, posCovariance

`unscentedKalmanFilter()` is given as follows. You don't need to change it.

In [ ]:
''' DO NOT CHANGE THE CODE BELOW '''
def unscentedKalmanFilter(position, 
                          deltaPos, 
                          magnetometerMeasurement, 
                          posCovariance, 
                          motionCovarianceUKF,
                          measurementCovarianceUKF,
                          lamdaWeights, 
                          weightsMean, 
                          weightsCovariance,
                          modelParameters):

    if modelParameters['NtimeStepUKF'] > 0:
        ''' State update '''
        position, posCovariance = dynamicUpdateUKF(position, 
                                                deltaPos, 
                                                posCovariance, 
                                                motionCovarianceUKF, 
                                                modelParameters)

    ''' Measurement update '''
    position, posCovariance = measurementUpdateUKF(position, 
                                                   magnetometerMeasurement, 
                                                   posCovariance, 
                                                   measurementCovarianceUKF,
                                                   lamdaWeights,
                                                   weightsMean,
                                                   weightsCovariance,
                                                   modelParameters)

    return position, posCovariance



#### Task 3

Be sure to set the **initial position** and the **prior** on it correctly  and a **suitable measurement covariance** as you did in assignment 1.

You could start with the following parameters
- `motionCovarianceUKF` = np.eye(3)*0.001**2 
- `measurementCovarianceUKF` = np.eye(3)*1.5**2

Make sure `posCovarianceUKF` is a 3d tensor with the shape of `[3,3,NtimeSteps]`, to save the position covariances for every time step.
To get better results, you might have to tune these. 

In [ ]:

''' Initialize the UKF '''
modelParameters['NtimeStepUKF'] = 0


motionCovarianceUKF = np.eye(3)*0.001**2
measurementCovarianceUKF = np.eye(3)*3.5**2

posEstUKF = np.zeros([3, N])
posEstUKF[:,0:1] = magnetometerInitialPosition

posCovarianceUKF = np.zeros([3, 3, N])
posCovarianceUKF[:, :, 0] = np.eye(3)*0.01**2

lamdaWeights, weightsMean, weightsCovariance = calculateSigmaWeights()
''' Run the UKF'''
for timendx in range(modelParameters['NtimeSteps']-1):
    (posEstUKF[:, timendx+1:timendx+2], 
     posCovarianceUKF[:, :, timendx+1]) = unscentedKalmanFilter(posEstUKF[:, timendx:timendx+1], 
                                               deltaMagnetometerPositions[:, timendx:timendx+1], 
                                               magnetometerMeasurements[:, timendx:timendx+1], 
                                               posCovarianceUKF[:, :, timendx],
                                               motionCovarianceUKF,
                                               measurementCovarianceUKF,
                                               lamdaWeights, 
                                               weightsMean, 
                                               weightsCovariance,
                                               modelParameters)
    modelParameters['NtimeStepUKF'] += 1



posCovarianceEKF = np.zeros((3, 3, modelParameters['NtimeSteps']))
posCovarianceEKF[:,:,0] = np.eye(3)*0


#### discussion

Elaborate on the effects of tuning the different variables:

- `motionCovarianceUKF`
- `measurementCovarianceUKF`
- The initial posiiton covariance `posCovarianceUKF[:, :, 0]`

Is it different from tuning the variables for the EKF?

## Answer

The effect of tunning the motion covariance Q is exactly the same because the dynamics in both cases are linear gaussian, so EKF and UKF propagate the mean and covariance identically in the prediction step. The effect of the measurement covariance R is also very similar because in both cases it regulates how much trust is given to the measurements. However, from a mathematical perspective, the effects vary a little. Particularly, the measurement update differs in how the nonlinearity of the magnetic field map is handled. Thus, the UKF achieves a slightly lower R, since a value of diag(3.5^2) already enables the UKF trajectory to approach the expected square trajectory. Meanwhile, for EKF, that value still made the estimated trajectory curve a little towards the center, so a value of diag(4^2) was chosen instead. 

The initial position covariance P0 has also the same effect as described for the EKF. However, as with R, there are some slight mathematical differences. In this case, the value of P0 chosen for EKF was too large for UKF, because it made the initial estimates drift too much from the expected trajectory. This indicated that a lower value is needed to trust the initial position more, since it aligns perfectly with the true initial position. A value of zero would therefore be correct, but a very low non-zero value was preferred because it is more realistic.

### compare EKF and UKF
#### Visualize the output

In [ ]:
helper.makeKalmanFilterPlots(deadReckoning, posEstEKF, posEstUKF, modelParameters)

#### Discussion
Elaborate on the two plots.
- Compare the results from the UKF with the EKF? Does one provide better results? Elaborate!
- Comment on the reason for choosing either EKF, UKF of PF for this non-linear estimation problem. Also relate this to the results from assignment 1. 
- In what type of estimation problems would you expect to see very different results for the EKF, UKF and PF? 

## Answer

- The results are very similar, but UKF yields slightly better results. Particularly on the top side, the estimated trajectory is visibly closer to the ground truth. It makes sense that they are similar because the dynamic model is the same for both cases. Since the best results were found by placing high trust in the dynamics, as reflected by the choices of the covariances, this means that the part of the filter that influences the results the most is identical in both cases, so a similar result is expected. Still, there is a slight difference because the UKF captures nonlinearities better than the EKF, which are present in the measurement model. This is consistent with what was learned in the lectures about UKF typically performing better than or equal to EKF.

- If we had infinite particles, then PF filtering would be the one that performs best. In practice, the number of particles is always finite, and the performance depends on how many particles can be used. This makes the PF the most computationally expensive method, followed by the UKF and then the EKF. This was noted when doing the assignments, since assignment 1 took longer to run than assignment 2. But, PF best captures nonlinearity. If the state dimension is not too high, a sufficiently large number of particles can be used while maintaining an acceptable computation time. If on top of that the model is too nonlinear or too multi-modal, then the PF becomes the best option. If the state dimension is higher and the distribution is not strongly multi-modal, then UKF has the advantage of reducing computation time and performing similarly, as was the case for this assignment. So, for this assignment UKF would be the best choice. EKF has an even lower computational cost, but for low dimensions the difference with UKF is negligible, and EKF performs notably worse for models which are too nonlinear. For this assignment the difference is not too high, as explained previously, but the difference in computation time is also negligible, so UKF still wins.

- As mentioned, if the problem is too multi-modal or too nonlinear then EKF would not perform well. In those cases, PF would be the optimal choice, with the caveat that if the dimensions are high then a lot of particles are needed to get accurate results, compromising computation efficiency. UKF would also not perform as well as PF in multi-modal cases. If things are not too multi-modal and there are high dimensions then UKF could also be used instead of PF, because it provides lower computational cost. That is the trade-off between PF and UKF. EKF works well if the model is approximately linear, since the linearization won't be too far off, and has the lowest computational cost among the three methods. 

<!-- TODO: DO WE NEED THE "PLAY AROUND WITH DIFFERENT PARAMETERS SESSION AS THE FIRST ASSIGMENT?" -->